# Práctica Guiada: Visualización Interactiva con Plotly

¡Bienvenido/a a la clase práctica de visualización de datos cinematográficos! En esta sesión, utilizaremos el poder y la interactividad de **Plotly** para explorar a fondo el dataset de `películas.csv`.

Aprenderás a construir gráficos dinámicos y profesionales que te permitirán descubrir:
1. **Distribución y éxito en taquilla** de películas por año y género.
2. **Relación entre Presupuestos e Ingresos** mediante sofisticados gráficos de dispersión.
3. **Segmentación por facetas** para comparar categorías de forma visual y rápida.
4. **Identificación y modelado de anomalías** financieras y de éxito con técnicas de Machine Learning.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1. Importar librerías principales
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética para gráficos
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

# 2. Cargar el dataset

import pandas as pd
data = pd.read_csv("/content/drive/MyDrive/Clases Diplomado/películas.csv")


# 3. Vista previa general de la estructura
display(data.head(5))
print(f"Dimensiones originales: {data.shape[0]} filas y {data.shape[1]} columnas.")

Comenzaremos asegurándonos de tener instalada la librería `plotly` para crear increíbles visualizaciones interactivas de nuestro catálogo de películas.

In [ ]:
pip install plotly

A continuación, importaremos las librerías necesarias, incluyendo `plotly.express` para una graficación más sencilla.

In [ ]:
import plotly.express as px

print('Plotly imported successfully!')

Comencemos analizando la distribución y el volumen de películas. Crearemos un gráfico de barras interactivo para explorar, por ejemplo, los ingresos o calificaciones de nuestras películas por año de estreno.

In [ ]:
# Seleccionar las 10 películas con mayor recaudación
top_10_movies = data.sort_values(by='revenue', ascending=False).head(10)

fig = px.bar(top_10_movies, x='title', y='revenue', title='Top 10 Películas con Mayor Recaudación')
fig.update_layout(xaxis_tickangle=-45, xaxis_title='Película', yaxis_title='Recaudación (Ingresos)')
fig.show()

Ahora visualicemos la evolución temporal de la industria del cine. Podemos agrupar las películas por su año de estreno para observar la tendencia de los ingresos promedio a lo largo de los años.

In [ ]:
# Convertir la fecha de estreno a formato datetime y extraer el año
data['Año_Estreno'] = pd.to_datetime(data['release_date'], errors='coerce').dt.year

# Filtrar para conservar solo años realistas (desde los inicios del cine hasta el año actual/próximo)
data_filtrada = data[(data['Año_Estreno'] >= 1880) & (data['Año_Estreno'] <= 2026)]

# Agrupar por el año de estreno corregido utilizando la columna 'revenue' (Ingresos)
yearly_stats = data_filtrada.groupby('Año_Estreno')['revenue'].mean().reset_index()

fig = px.line(yearly_stats, x='Año_Estreno', y='revenue', markers=True, title='Evolución del Promedio de Ingresos por Año (Datos Filtrados)')
fig.update_layout(xaxis_title='Año de Estreno', yaxis_title='Promedio de Ingresos')
fig.show()

Enriqueceremos la visualización con información interactiva al pasar el cursor. En este gráfico de dispersión cruzaremos el Presupuesto contra los Ingresos de las películas, mostrando el Título en los detalles.

In [ ]:
# Filtrar películas con calificación mayor que 0 y seleccionar las 100 con mayores presupuestos
peliculas_con_voto = data[data['vote_average'] > 0]
top_100_budget = peliculas_con_voto.sort_values(by='budget', ascending=False).head(100)

fig = px.scatter(
    top_100_budget,
    x='budget',
    y='vote_average',
    hover_name='title',
    title='Relación Presupuesto vs Calificación (Top 100 Presupuesto con Calificación > 0)'
)
fig.update_layout(xaxis_title='Presupuesto', yaxis_title='Calificación (IMDB)')
fig.show()

Este análisis nos permite identificar fácilmente qué películas fueron éxitos rotundos en taquilla y cuáles tuvieron presupuestos elevados pero baja recaudación.

Veamos cómo se distribuyen los géneros principales en nuestra muestra utilizando barras agrupadas para comparar presupuestos e ingresos.

In [ ]:
# Filtrar películas con presupuesto > 0 y seleccionar las top 100 con más ingresos
peliculas_con_presupuesto = data[data['budget'] > 0]
top_100_revenue = peliculas_con_presupuesto.sort_values(by='revenue', ascending=False).head(100)

# Agrupar por la columna 'adult' (¿Es para adultos?) y calcular los promedios de presupuesto e ingresos
adult_group_data = top_100_revenue.groupby('adult')[['budget', 'revenue']].mean().reset_index()

# Mapear los valores booleanos a etiquetas más descriptivas para el gráfico
adult_group_data['Tipo de Público'] = adult_group_data['adult'].map({True: 'Para Adultos', False: 'Todo Público'})

fig = px.bar(
    adult_group_data,
    x='Tipo de Público',
    y=['budget', 'revenue'],
    barmode='group',
    title='Promedio de Presupuesto e Ingresos por Tipo de Público (Top 100 Ingresos con Presupuesto > 0)'
)
fig.update_layout(xaxis_title='Tipo de Público', yaxis_title='Monto promedio (USD)')
fig.show()

Plotly permite segmentar la información en subpaneles independientes (facetas) para realizar comparaciones directas por categorías como el Género.

In [ ]:
# Filtrar películas con ingresos > 0 y calificación > 0
peliculas_filtradas = data[(data['revenue'] > 0) & (data['vote_average'] > 0)]

# Seleccionar las top 100 con mayores ingresos
top_100_rev_voto = peliculas_filtradas.sort_values(by='revenue', ascending=False).head(100)

fig = px.scatter(
    top_100_rev_voto,
    x='vote_average',
    y='revenue',
    facet_col='adult',
    hover_name='title',
    title='Calificación vs Ingresos por Tipo de Público (Top 100 con Ingresos y Votos > 0)'
)
fig.update_layout(xaxis_title='Calificación', yaxis_title='Ingresos')
fig.show()

Personalizar los colores es sumamente sencillo en Plotly. Utilizaremos paletas vibrantes para diferenciar nuestros géneros cinematográficos de forma llamativa.

In [ ]:
# Filtrar películas con duración > 0 y calificación > 0
peliculas_filtradas_dur_voto = data[(data['runtime'] > 0) & (data['vote_average'] > 0)]

# Seleccionar las top 100 películas con mayor calificación
top_10000_calificadas = peliculas_filtradas_dur_voto.sort_values(by='vote_average', ascending=False).head(10000)

fig = px.scatter(
    top_10000_calificadas,
    x='runtime',
    y='vote_average',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hover_name='title',
    title='Duración vs Calificación (Top 10000 Mejor Calificadas)'
)
fig.update_layout(xaxis_title='Duración (minutos)', yaxis_title='Calificación (IMDB)')
fig.show()

Para un estilo cinematográfico elegante, podemos aplicar una plantilla oscura a nuestras visualizaciones.

In [ ]:
fig = px.scatter(
    data.head(200),
    x='budget',
    y='revenue',
    color='vote_average',
    template='plotly_dark',
    hover_name='title',
    title='Análisis Cinematográfico (Tema Oscuro)'
)
fig.update_layout(xaxis_title='Presupuesto', yaxis_title='Ingresos')
fig.show()

Es muy útil añadir líneas de referencia como umbrales críticos de calidad cinematográfica (por ejemplo, una calificación aprobatoria de 7.0).

In [ ]:
fig = px.scatter(
    data.head(20000),
    x='runtime',
    y='vote_average',
    hover_name='title',
    title='Calificación de Películas con Umbral de Referencia'
)
fig.add_hline(y=7.0, line_dash='dash', line_color='red', annotation_text='Calificación Deseada')
fig.update_layout(
    xaxis_title='Duración (minutos)',
    yaxis_title='Calificación (IMDB)'
)
fig.show()

Los mapas de calor son excelentes para visualizar correlaciones entre variables numéricas del dataset como presupuesto, ingresos, duración y calificación.

In [ ]:
corr = data.select_dtypes(include='number').corr()
fig = px.imshow(corr, text_auto=True, color_continuous_scale='Viridis', title='Matriz de Correlación del Dataset de Películas')
fig.show()

Dado que las películas cuentan con años o fechas de lanzamiento, Plotly permite analizar la escala temporal de manera óptima.

In [ ]:
# Filtrar registros con presupuesto > 0, ingresos > 0, calificación > 0 y año válido
condiciones = (
    (data['budget'] > 0) &
    (data['revenue'] > 0) &
    (data['vote_average'] > 0) &
    (data['Año_Estreno'] >= 1880) &
    (data['Año_Estreno'] <= 2026)
)
data_filtrada_temporal = data[condiciones].head(10000)

fig = px.scatter(
    data_filtrada_temporal,
    x='Año_Estreno',
    y='revenue',
    hover_name='title',
    title='Evolución Temporal de Ingresos (Primeros 10k Registros con Presupuesto, Ingreso y Calificación > 0)'
)
fig.update_layout(xaxis_title='Año de Estreno', yaxis_title='Ingresos (USD)')
fig.show()

In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

# 1. Aplicar filtros solicitados: presupuesto > 0, ingresos > 0, calificación > 0 y tomar primeros 10000
condiciones = (data['budget'] > 0) & (data['revenue'] > 0) & (data['vote_average'] > 0)
data_ml = data[condiciones].head(10000).copy()

# 2. Calcular Rentabilidad sobre los datos filtrados
data_ml['Rentabilidad'] = data_ml['revenue'] / data_ml['budget']

# 3. Definir características utilizando las columnas correctas en inglés
features = ['budget', 'revenue', 'vote_average', 'Rentabilidad']
X = data_ml[features].fillna(0)
X = X.replace([np.inf, -np.inf], 0)

# 4. Entrenar el Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
data_ml['Anomaly_Score'] = iso_forest.fit_predict(X)

# 5. Mapear estado de anomalía
data_ml['Status'] = data_ml['Anomaly_Score'].map({1: 'Normal', -1: 'Anomalía'})

print("Resumen de anomalías en el catálogo filtrado de películas:")
display(data_ml['Status'].value_counts())

# Mostrar ejemplos de anomalías con las columnas correctas del dataset
anomalies = data_ml[data_ml['Status'] == 'Anomalía']
print("\nEjemplos de películas catalogadas como anomalías (Presupuestos o Retornos inusuales):")
display(anomalies[['title', 'budget', 'revenue', 'vote_average', 'Status']].head())

In [ ]:
fig = px.scatter(
    data_ml,
    x='budget',
    y='revenue',
    color='Status',
    color_discrete_map={'Normal': '#636EFA', 'Anomalía': '#EF553B'},
    title='Detección de Anomalías en Películas (Presupuesto vs Ingresos - Isolation Forest)',
    hover_data=['title', 'vote_average']
)
fig.update_layout(
    xaxis_title='Presupuesto (USD)',
    yaxis_title='Ingresos (USD)'
)
fig.show()

Finalmente, puedes exportar cualquier figura de Plotly a un archivo HTML autónomo. Este archivo será interactivo y se podrá compartir fácilmente, empaquetando todo el JavaScript necesario de Plotly.

In [ ]:
fig.write_html('dashboard_peliculas.html', include_plotlyjs='cdn')
print("¡Dashboard de películas exportado correctamente a 'dashboard_peliculas.html'!")